In [ ]:
import os
from pathlib import Path
# Ensure CWD is repo root so relative paths and `tools.*` imports resolve.
if Path.cwd().name == "notebooks":
    os.chdir("..")


# NER for decicontas.br — Multi-Model & Multi-Technique Evaluation

This notebook evaluates **18 LLMs** across **6 provider families** using **OpenRouter** as a unified API gateway, combined with **8 prompt engineering techniques** on the decicontas.br dataset.

## Models

| Family | Model | OpenRouter ID | Context |
|--------|-------|---------------|---------|
| OpenAI 4.x | GPT-3.5 | `openai/gpt-3.5-turbo` | 16K |
| OpenAI 4.x | GPT-4 Turbo | `openai/gpt-4-turbo` | 128K |
| OpenAI 4.x | GPT-4o | `openai/gpt-4o` | 128K |
| OpenAI 4.x | GPT-4.1 | `openai/gpt-4.1` | 1M |
| OpenAI 4.x | GPT-4.1 Mini | `openai/gpt-4.1-mini` | 1M |
| OpenAI 4.x | GPT-4.1 Nano | `openai/gpt-4.1-nano` | 1M |
| OpenAI 5.x | GPT-5 | `openai/gpt-5` | 128K |
| Anthropic | Claude Sonnet 4.5 | `anthropic/claude-sonnet-4-5` | 200K |
| Anthropic | Claude Haiku 4.5 | `anthropic/claude-haiku-4-5` | 200K |
| Anthropic | Claude Sonnet 4.6 | `anthropic/claude-sonnet-4-6` | 1M |
| Google | Gemini 2.5 Pro | `google/gemini-2.5-pro` | 1M |
| Google | Gemini 2.5 Flash | `google/gemini-2.5-flash` | 1M |
| DeepSeek | DeepSeek V3 | `deepseek/deepseek-chat-v3-0324` | 128K |
| DeepSeek | DeepSeek R1 | `deepseek/deepseek-r1` | 128K |
| Qwen | Qwen 3 235B | `qwen/qwen3-235b` | 128K |
| Meta | LLaMA 4 Maverick | `meta-llama/llama-4-maverick` | 1M |
| Mistral | Mistral Medium 3 | `mistralai/mistral-medium-3` | 128K |
| Mistral | Mistral Small 3.2 | `mistralai/mistral-small-3.2` | 128K |

## Prompt Engineering Techniques
1. **Few-Shot (baseline)** — In-context learning with curated examples
2. **Chain-of-Thought (CoT)** — Step-by-step reasoning before extraction
3. **Negative Examples** — Showing what NOT to extract
4. **Role Prompting** — Specialized legal auditor persona
5. **Structured Definitions** — Formal entity definitions in prompt
6. **Two-Stage Pipeline** — Classification then extraction
7. **Dynamic Few-Shot** — Embedding-based example selection
8. **Self-Refinement** — Extract then verify/correct


In [30]:
import spacy
import pprint
import time
import json
import os
import pickle
import wandb
import re

import pandas as pd
import numpy as np

from pathlib import Path

from collections import defaultdict, Counter
from sklearn.metrics import precision_recall_fscore_support, classification_report

from tqdm import tqdm
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI, OpenAIEmbeddings, AzureChatOpenAI

from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

from concurrent.futures import ThreadPoolExecutor, as_completed

from pydantic import ValidationError

from tools.dataset import get_decicontas_df
from tools.prompt import (
    generate_few_shot_ner_prompts,
    FEW_SHOT_NER_PROMPT,
)
from tools.fewshot import TOOL_USE_EXAMPLES, get_formatted_messages_from_examples
from tools.schema import NERDecisao

WANDB_PROJECT = 'decicontas.br'
WANDB_ENTITY = 'eduardoplima-imd'

load_dotenv()

True

# 1. OpenRouter Configuration

All models are accessed through [OpenRouter](https://openrouter.ai/) using a single API key via `ChatOpenAI` with a custom `base_url`. This eliminates the need for separate provider SDKs (Azure, Anthropic, Google, etc.).

Set `OPENROUTER_API_KEY` in your `.env` file.


In [31]:
# ============================================================================
# OPENROUTER CONFIGURATION
# ============================================================================

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
OPENAI_API_VERSION = os.getenv("OPENAI_API_VERSION")
AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")

# Mapeamento: model_id do OpenRouter -> deployment name na Azure
AZURE_DEPLOYMENTS = {
    "openai/gpt-3.5-turbo": "gpt-35-turbo",
    "openai/gpt-4o": "gpt-4o",
    "openai/gpt-5.4": "gpt-5.4",
    "openai/gpt-5.4-mini": "gpt-5.4-mini",
    "openai/gpt-5.4-nano": "gpt-5.4-nano",
}


def make_llm(model_id: str, **kwargs) -> ChatOpenAI:
    kwargs.setdefault("max_tokens", 4096)
    
    if model_id in AZURE_DEPLOYMENTS:
        return AzureChatOpenAI(
            azure_deployment=AZURE_DEPLOYMENTS[model_id],
            azure_endpoint=AZURE_OPENAI_ENDPOINT,
            api_key=AZURE_OPENAI_API_KEY,
            api_version=OPENAI_API_VERSION,
            **kwargs,
        )
    
    extra_body = kwargs.pop("model_kwargs", {}).get("extra_body", {})
    
    # Força provider nativo para evitar proxies lentos/quebrados
    if "deepseek" in model_id:
        extra_body["provider"] = {"order": ["DeepSeek"]}
    
    return ChatOpenAI(
        model=model_id,
        openai_api_key=OPENROUTER_API_KEY,
        openai_api_base=OPENROUTER_BASE_URL,
        default_headers={
            "HTTP-Referer": "https://github.com/decicontas",
            "X-Title": "decicontas-ner",
        },
        model_kwargs={"extra_body": extra_body} if extra_body else {},
        **kwargs,
    )

# Modelos que NÃO suportam response_format: json_schema
NO_STRUCTURED_OUTPUT = {"openai/gpt-3.5-turbo"}


# DEPOIS (corrigido)
def make_extractor(model_id: str):
    llm = make_llm(model_id)
    if model_id in NO_STRUCTURED_OUTPUT:
        llm_with_json = llm.bind(response_format={"type": "json_object"})
        return llm_with_json.with_structured_output(NERDecisao, method="json_mode", include_raw=False)
    return llm.with_structured_output(NERDecisao, include_raw=False, method="function_calling")

print(f"OpenRouter API Key configured: {'✓' if OPENROUTER_API_KEY else '✗'}")

OpenRouter API Key configured: ✓


In [32]:
# ============================================================================
# MODEL REGISTRY
# ============================================================================

MODEL_REGISTRY = {
    "gpt-35-turbo":       "openai/gpt-3.5-turbo",
    # --- OpenAI 4.x (baseline) ---
    "gpt-4o":               "openai/gpt-4o",
    # --- OpenAI 5.x ---
    "gpt-5.4":              "openai/gpt-5.4",
    "gpt-5.4-mini":         "openai/gpt-5.4-mini",
    "gpt-5.4-nano":         "openai/gpt-5.4-nano",
    # --- Anthropic ---
    "claude-sonnet-4-5":    "anthropic/claude-sonnet-4-5",
    "claude-haiku-4-5":     "anthropic/claude-haiku-4-5",
    "claude-sonnet-4-6":    "anthropic/claude-sonnet-4-6",
    # --- Google ---
    "gemini-2.5-pro":       "google/gemini-2.5-pro",
    "gemini-2.5-flash":     "google/gemini-2.5-flash",
    # --- DeepSeek ---
    "deepseek-v3":          "deepseek/deepseek-chat-v3-0324",
    "deepseek-r1":          "deepseek/deepseek-r1",
    # --- Qwen ---
    "qwen3-235b":           "qwen/qwen3-235b-a22b",
    # --- Meta ---
    "llama-4-maverick":     "meta-llama/llama-4-maverick",
    # --- Mistral ---
    #"mistral-medium-3":     "mistralai/mistral-medium-3",
    "mistral-small-3.2":    "mistralai/mistral-small-3.2-24b-instruct",
}

# Create extractors for all models
MODELS = []
model_dict = {}
for name, openrouter_id in MODEL_REGISTRY.items():
    extractor = make_extractor(openrouter_id)
    MODELS.append((name, extractor))
    model_dict[name] = extractor

models_names = [m[0] for m in MODELS]
print(f"Models configured: {len(MODELS)}")
for name in models_names:
    print(f"  • {name} → {MODEL_REGISTRY[name]}")

Models configured: 15
  • gpt-35-turbo → openai/gpt-3.5-turbo
  • gpt-4o → openai/gpt-4o
  • gpt-5.4 → openai/gpt-5.4
  • gpt-5.4-mini → openai/gpt-5.4-mini
  • gpt-5.4-nano → openai/gpt-5.4-nano
  • claude-sonnet-4-5 → anthropic/claude-sonnet-4-5
  • claude-haiku-4-5 → anthropic/claude-haiku-4-5
  • claude-sonnet-4-6 → anthropic/claude-sonnet-4-6
  • gemini-2.5-pro → google/gemini-2.5-pro
  • gemini-2.5-flash → google/gemini-2.5-flash
  • deepseek-v3 → deepseek/deepseek-chat-v3-0324
  • deepseek-r1 → deepseek/deepseek-r1
  • qwen3-235b → qwen/qwen3-235b-a22b
  • llama-4-maverick → meta-llama/llama-4-maverick
  • mistral-small-3.2 → mistralai/mistral-small-3.2-24b-instruct


/var/folders/39/p0t9g2qn1gbcvnwqyz79d34c0000gn/T/ipykernel_91029/2213991137.py:36: UserWarning: Parameters {'extra_body'} should be specified explicitly. Instead they were passed in as part of `model_kwargs` parameter.
  extractor = make_extractor(openrouter_id)
/var/folders/39/p0t9g2qn1gbcvnwqyz79d34c0000gn/T/ipykernel_91029/2213991137.py:36: UserWarning: Parameters {'extra_body'} should be specified explicitly. Instead they were passed in as part of `model_kwargs` parameter.
  extractor = make_extractor(openrouter_id)


# 2. Prompt Engineering Techniques

All 8 techniques: prompt templates and helper functions.


In [ ]:
from tools.prompt_engineering import (
    COT_NER_PROMPT,
    NEGATIVE_EXAMPLES_PROMPT,
    ROLE_PROMPT,
    DEFINITIONS_PROMPT,
    DocumentClassification,
    CLASSIFICATION_PROMPT,
    VERIFICATION_PROMPT,
    TECHNIQUE_PROMPTS,
    generate_prompt_for_technique,
    two_stage_ner,
    extract_with_verification,
    make_embeddings_model,
    dynamic_few_shot_selection,
    generate_dynamic_few_shot_prompt,
)

In [ ]:
# Classifiers for two-stage (cheap/fast models) — notebook-specific: depends on make_llm/OpenRouter.
CLASSIFIER_MODELS = {
    "gpt-4.1-mini":      make_llm("openai/gpt-4.1-mini").with_structured_output(DocumentClassification),
    "claude-haiku-4-5":  make_llm("anthropic/claude-haiku-4-5").with_structured_output(DocumentClassification),
    "gemini-2.5-flash":  make_llm("google/gemini-2.5-flash").with_structured_output(DocumentClassification),
    "deepseek-v3":       make_llm("deepseek/deepseek-chat-v3-0324").with_structured_output(DocumentClassification),
    "gpt-5.4-mini":      make_llm("openai/gpt-5.4-mini").with_structured_output(DocumentClassification),
    "gpt-5.4-nano":      make_llm("openai/gpt-5.4-nano").with_structured_output(DocumentClassification),
    "qwen3-235b":        make_llm("qwen/qwen3-235b").with_structured_output(DocumentClassification),
    "llama-4-maverick":  make_llm("meta-llama/llama-4-maverick").with_structured_output(DocumentClassification),
    "mistral-small-3.2": make_llm("mistralai/mistral-small-3.2").with_structured_output(DocumentClassification),
}

# Embeddings for dynamic few-shot.
embeddings_model = make_embeddings_model()

# 3. Dataset Loading


In [35]:
nlp = spacy.load("pt_core_news_sm")
df_decicontas = get_decicontas_df()

def tem_rotulo(row):
    for annotation in row["annotations"]:
        if annotation["result"]:
            return True
    return False

df_decicontas.loc[:, "tem_rotulo"] = df_decicontas.apply(tem_rotulo, axis=1)
print(f"Total rows: {len(df_decicontas)}, with labels: {df_decicontas['tem_rotulo'].sum()}")

Total rows: 866, with labels: 229


# 4. Experiment Configuration

- **Simple techniques** (prompt-only, no extra cost): run on ALL 18 models
- **Complex techniques** (extra API calls): run on a representative subset


In [36]:
# ============================================================================
# EXPERIMENT MATRIX
# ============================================================================

SIMPLE_TECHNIQUES = ["few_shot", "cot", "negative_examples", "role_prompting", "definitions"]

# Representative models for complex techniques (one strong per family)
REPRESENTATIVE_MODELS = [
    "gpt-4-turbo", "gpt-4.1-mini", "gpt-5",
    "claude-sonnet-4-5", "gemini-2.5-pro",
    "deepseek-v3", "deepseek-r1",
    "qwen3-235b", "llama-4-maverick",
]

COMPLEX_TECHNIQUES_CONFIG = {
    "two_stage": {
        "models": REPRESENTATIVE_MODELS,
        "classifier_map": {
            "gpt-4-turbo": "gpt-4.1-mini",
            "gpt-4.1-mini": "gpt-4.1-mini",
            "gpt-5": "gpt-4.1-mini",
            "claude-sonnet-4-5": "claude-haiku-4-5",
            "gemini-2.5-pro": "gemini-2.5-flash",
            "deepseek-v3": "deepseek-v3",
            "deepseek-r1": "deepseek-v3",
            "qwen3-235b": "gpt-4.1-mini",
            "llama-4-maverick": "gpt-4.1-mini",
        }
    },
    "dynamic_few_shot": {"models": REPRESENTATIVE_MODELS},
    "self_refinement": {"models": REPRESENTATIVE_MODELS},
}

COMPLEX_TECHNIQUES_CONFIG["two_stage"]["classifier_map"].update({
    "gpt-5.4-mini":      "gpt-5.4-mini",
    "gpt-5.4-nano":      "gpt-5.4-nano",
    "gemini-2.5-flash":  "gemini-2.5-flash",
    "deepseek-v3":       "deepseek-v3",
    "qwen3-235b":        "qwen3-235b",
    "llama-4-maverick":  "llama-4-maverick",
    "mistral-small-3.2": "mistral-small-3.2",
})

# Build full experiment list
experiments = []
for technique in SIMPLE_TECHNIQUES:
    for model_name in models_names:
        experiments.append((model_name, technique))

for tech_name, config in COMPLEX_TECHNIQUES_CONFIG.items():
    for model_name in config["models"]:
        experiments.append((model_name, tech_name))

n_simple = len(SIMPLE_TECHNIQUES) * len(MODELS)
n_complex = len(experiments) - n_simple
print(f"Total experiments: {len(experiments)}")
print(f"  Simple: {len(SIMPLE_TECHNIQUES)} techniques × {len(MODELS)} models = {n_simple}")
print(f"  Complex: {n_complex}")

Total experiments: 102
  Simple: 5 techniques × 15 models = 75
  Complex: 27


## Cost Analysis

In [37]:
NUM_DOCS = 866

# Ajuste com base numa amostra real
AVG_INPUT_TOKENS = 8630   # tokens médios por request (prompt + texto)
AVG_OUTPUT_TOKENS =60    # tokens médios de resposta

# Preços OpenRouter ($/1M tokens) — abril 2026
PRICING = {
    "gpt-4o":              {"input": 2.50, "output": 10.00},
    "gpt-5.4":             {"input": 2.50, "output": 20.00},
    "gpt-5.4-mini":        {"input": 0.40, "output": 1.60},
    "gpt-5.4-nano":        {"input": 0.10, "output": 0.40},
    "claude-sonnet-4-5":   {"input": 3.00, "output": 15.00},
    "claude-haiku-4-5":    {"input": 1.00, "output": 5.00},
    "claude-sonnet-4-6":   {"input": 3.00, "output": 15.00},
    "gemini-2.5-pro":      {"input": 1.25, "output": 10.00},
    "gemini-2.5-flash":    {"input": 0.15, "output": 0.60},
    "deepseek-v3":         {"input": 0.32, "output": 0.89},
    "deepseek-r1":         {"input": 0.55, "output": 2.19},
    "qwen3-235b":          {"input": 0.00, "output": 0.00},  # free no OpenRouter
    "llama-4-maverick":    {"input": 0.00, "output": 0.00},  # free
    "mistral-medium-3":    {"input": 0.40, "output": 2.00},
    "mistral-small-3.2":   {"input": 0.10, "output": 0.30},
}

NUM_TECHNIQUES = 4  # few_shot, two_stage, dynamic_few_shot, self_refinement

print(f"{'Modelo':<25} {'$/run':>8} {'$/todas técnicas':>18}")
print("-" * 55)

# Todos os modelos com few_shot (1 técnica) + top modelos com 4 técnicas
PLAN = {
    "gpt-4o":            1,
    "gemini-2.5-pro":    1,
    "gpt-5.4-mini":      4,
    "gpt-5.4-nano":      4,
    "claude-haiku-4-5":  1,
    "gemini-2.5-flash":  4,
    "deepseek-v3":       4,
    "qwen3-235b":        4,
    "llama-4-maverick":  4,
    "mistral-small-3.2": 4,
}

total = 0
for model, n_tech in PLAN.items():
    p = PRICING[model]
    cost = (AVG_INPUT_TOKENS * p["input"] + AVG_OUTPUT_TOKENS * p["output"]) / 1_000_000 * NUM_DOCS * n_tech
    total += cost
    print(f"{model:<25} {n_tech} técnicas  ${cost:.2f}")
print(f"\nTOTAL: ${total:.2f}")

print("-" * 55)
print(f"{'TOTAL':<25} {'':>8} ${total:>17.2f}")

Modelo                       $/run   $/todas técnicas
-------------------------------------------------------
gpt-4o                    1 técnicas  $19.20
gemini-2.5-pro            1 técnicas  $9.86
gpt-5.4-mini              4 técnicas  $12.29
gpt-5.4-nano              4 técnicas  $3.07
claude-haiku-4-5          1 técnicas  $7.73
gemini-2.5-flash          4 técnicas  $4.61
deepseek-v3               4 técnicas  $9.75
qwen3-235b                4 técnicas  $0.00
llama-4-maverick          4 técnicas  $0.00
mistral-small-3.2         4 técnicas  $3.05

TOTAL: $69.57
-------------------------------------------------------
TOTAL                              $            69.57


# 5. Evaluation Metrics

Same dual-metric framework:
- **Token-level**: P/R/F1 (ignoring B-/I-, excluding 'O')
- **Span-level IoU ≥ 0.5**: Aggregate and per-label


In [38]:
from tools.ner_metrics import (
    DICT_LABELS,
    convert_pred_to_golden_format,
    _strip_bio,
    compute_iou_score,
    calculate_metrics,
    evaluate_results,
)


# 6. Batch Inference

For each (model, technique) pair, run NER on the full dataset and save results.


In [39]:
os.makedirs("dataset/results", exist_ok=True)

def get_result_path(model_name, technique):
    return f"dataset/results/models_results_decicontas_{model_name}_{technique}.json"

def clean_json_response(text):
    return re.sub(r'^```(?:json)?\s*|\s*```$', '', text.strip())

def run_experiment(model_name, technique, df):
    result_path = get_result_path(model_name, technique)
    
    if os.path.exists(result_path):
        existing = pd.read_json(result_path)
        if len(existing) >= len(df):
            print(f"  Already complete: {model_name} / {technique}")
            return existing
    
    extractor = model_dict[model_name]
    results = []
    errors = 0

    for index, row in tqdm(df.iterrows(), desc=f"{model_name}/{technique}", total=len(df), unit="doc"):
        text = row['data']['text']
        golden = [r['value'] for r in row['annotations'][0]['result']]
        
        success = False
        result = None
        retries = 0
        
        while not success and retries < 10:
            try:
                if technique in SIMPLE_TECHNIQUES:
                    prompt = generate_prompt_for_technique(text, technique)
                    result = extractor.invoke(prompt)
                elif technique == "dynamic_few_shot":
                    prompt = generate_dynamic_few_shot_prompt(text, embeddings_model, k=5)
                    result = extractor.invoke(prompt)
                elif technique == "two_stage":
                    clf_name = COMPLEX_TECHNIQUES_CONFIG["two_stage"]["classifier_map"][model_name]
                    classifier = CLASSIFIER_MODELS[clf_name]
                    result = two_stage_ner(classifier, extractor, text, generate_few_shot_ner_prompts)
                    if result is None:
                        result = NERDecisao(multas=[], obrigacoes=[], recomendacoes=[], ressarcimentos=[])
                elif technique == "self_refinement":
                    result = extract_with_verification(extractor, extractor, text, generate_few_shot_ner_prompts)
                success = True
            except ValidationError:
                retries += 1
                errors += 1
                time.sleep(2)
            except Exception as e:
                retries += 1
                if "is not a valid model ID" in str(e):
                    raise RuntimeError(f"Modelo inválido: {model_name}") from e
                print(f"  Error at index {index} (attempt {retries}): {e}")
                time.sleep(5 * retries)  # backoff progressivo
        
        if not success:
            errors += 1
            result = NERDecisao(multas=[], obrigacoes=[], recomendacoes=[], ressarcimentos=[])
            print(f"  FALHA DEFINITIVA doc {index} após {retries} tentativas")
        
        results.append({
            'index': index,
            'text': text,
            'pred': result.model_dump() if hasattr(result, 'model_dump') else result,
            'golden': golden,
            'model': model_name,
            'technique': technique,
        })
    
    print(f"\n  Concluído: {len(results)} docs, {errors} erros")
    df_results = pd.DataFrame(results)
    df_results.to_json(result_path, orient="records", force_ascii=False, indent=2)
    return df_results

In [41]:
CHECKPOINT_DIR = Path("dataset/results/checkpoints").resolve()
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

def get_checkpoint_path(model_name: str, technique: str) -> Path:
    safe_name = model_name.replace("/", "_").replace(".", "-")
    return CHECKPOINT_DIR / f"{safe_name}__{technique}.pkl"

PLAN = {
    #"gpt-4o":            ["few_shot"],
    #"gemini-2.5-pro":    ["few_shot"],
    #"claude-haiku-4-5":  ["few_shot"],
    #"gpt-5.4":           ["few_shot"],
    #"gpt-5.4-mini":      ["few_shot", "dynamic_few_shot", "two_stage", "self_refinement"],
    "gpt-5.4-mini":      ["cot"],
    "gpt-5.4-nano":      ["cot"],
    #"gemini-2.5-flash":  ["few_shot", "dynamic_few_shot", "two_stage", "self_refinement"],
    "gemini-2.5-flash":  ["cot"],
    #"deepseek-v3":       ["few_shot"],
    "deepseek-v3":       ["cot"],
   #"qwen3-235b":        ["few_shot"],
    #"llama-4-maverick":  ["few_shot"],
    #"mistral-small-3.2": ["few_shot", "dynamic_few_shot", "two_stage", "self_refinement"],
}

experiments = [(model, tech) for model, techs in PLAN.items() for tech in techs]

error_log = {}

for model_name, technique in experiments:
    ckpt_path = get_checkpoint_path(model_name, technique)
    
    if ckpt_path.exists():
        print(f"[SKIP] {model_name} / {technique} — checkpoint encontrado")
        continue
    
    print(f"\n{'='*60}")
    print(f"Running: {model_name} / {technique}")
    print(f"{'='*60}")
    
    try:
        result = run_experiment(model_name, technique, df_decicontas)
        with open(ckpt_path, "wb") as f:
            pickle.dump(result, f)
        print(f"[SAVED] {ckpt_path}")
    except Exception as e:
        if "is not a valid model ID" in str(e):
            print(f"[SKIP ALL] {model_name} — modelo inválido no OpenRouter")
            # Pula todas as técnicas desse modelo
            continue
        print(f"[ERROR] {model_name} / {technique}: {e}")
        error_log[(model_name, technique)] = str(e)

if error_log:
    print(f"\n{'='*60}")
    print(f"Total de falhas: {len(error_log)}")
    for (m, t), err in error_log.items():
        print(f"  - {m} / {t}: {err[:100]}")



Running: gpt-5.4-mini / cot


gpt-5.4-mini/cot: 100%|██████████| 866/866 [30:59<00:00,  2.15s/doc]



  Concluído: 866 docs, 0 erros
[SAVED] /Users/eduardo/Dev/decicontas.br/dataset/results/checkpoints/gpt-5-4-mini__cot.pkl

Running: gpt-5.4-nano / cot


gpt-5.4-nano/cot: 100%|██████████| 866/866 [31:46<00:00,  2.20s/doc]



  Concluído: 866 docs, 0 erros
[SAVED] /Users/eduardo/Dev/decicontas.br/dataset/results/checkpoints/gpt-5-4-nano__cot.pkl

Running: gemini-2.5-flash / cot


gemini-2.5-flash/cot: 100%|██████████| 866/866 [27:37<00:00,  1.91s/doc] 



  Concluído: 866 docs, 0 erros
[SAVED] /Users/eduardo/Dev/decicontas.br/dataset/results/checkpoints/gemini-2-5-flash__cot.pkl

Running: deepseek-v3 / cot


deepseek-v3/cot: 100%|██████████| 866/866 [1:29:21<00:00,  6.19s/doc]  


  Concluído: 866 docs, 0 erros
[SAVED] /Users/eduardo/Dev/decicontas.br/dataset/results/checkpoints/deepseek-v3__cot.pkl


# 6. Computing Metrics for All Experiments


In [55]:
import math

In [61]:
def fill_empty_pred(x):
    if not isinstance(x, dict) and math.isnan(x):
        return {
            "multas": [],
            "obrigacoes": [],
            "recomendacoes": [],
            "ressarcimentos": []
        }
    else:
        return x

df_exp['pred'] = df_exp['pred'].apply(fill_empty_pred)

In [62]:
df_exp

,index,text,pred,golden,model,technique
0,0,DECIDEM os Conselheiros do Tribunal de Contas ...,"{'multas': [], 'ressarcimentos': [], 'obrigaco...",[],deepseek-v3,cot
1,1,DECIDEM os Conselheiros do Tribunal de Contas ...,{'multas': [{'descricao_multa': 'MULTA diária ...,[],deepseek-v3,cot
2,2,DECIDEM os Conselheiros do Tribunal de Contas ...,"{'multas': [], 'ressarcimentos': [], 'obrigaco...",[],deepseek-v3,cot
3,3,"Vistos, relatados e discutidos estes autos, ac...","{'multas': [], 'obrigacoes': [], 'recomendacoe...",[],deepseek-v3,cot
4,4,DECIDEM os Conselheiros do Tribunal de Contas ...,"{'multas': [], 'obrigacoes': [], 'recomendacoe...",[],deepseek-v3,cot
...,...,...,...,...,...,...
861,861,"Vistos, relatados e discutidos estes autos do ...","{'multas': [], 'ressarcimentos': [{'descricao_...","[{'start': 910, 'end': 1352, 'text': 'ressarci...",deepseek-v3,cot
862,862,"Vistos, relatados e discutidos estes autos de ...","{'multas': [], 'ressarcimentos': [{'descricao_...","[{'start': 493, 'end': 683, 'text': 'ressarcim...",deepseek-v3,cot
863,863,"Vistos, relatados e discutidos estes autos da ...",{'multas': [{'descricao_multa': 'Multa no valo...,"[{'start': 887, 'end': 1410, 'text': 'Ressarci...",deepseek-v3,cot
864,864,"Vistos, relatados e discutidos estes autos do ...",{'multas': [{'descricao_multa': 'multa total d...,"[{'start': 588, 'end': 1249, 'text': 'ressarci...",deepseek-v3,cot


In [63]:
df_exp.to_json("dataset/results/models_results_decicontas_deepseek-v3_cot.json", orient="records", force_ascii=False, indent=2)

In [59]:
df_exp['pred'].apply(lambda x: math.isnan(x) if isinstance(x, float) else False).sum()

43

In [56]:
df_exp['pred'].apply(lambda x: {"multas": [], "obrigacoes": [], "recomendacoes": [], "ressarcimentos": []} if math.isnan(x) else x)

TypeError: must be real number, not dict

In [51]:
df_exp

,index,text,pred,golden,model,technique
0,0,DECIDEM os Conselheiros do Tribunal de Contas ...,"{'multas': [], 'ressarcimentos': [], 'obrigaco...",[],deepseek-v3,cot
1,1,DECIDEM os Conselheiros do Tribunal de Contas ...,{'multas': [{'descricao_multa': 'MULTA diária ...,[],deepseek-v3,cot
2,2,DECIDEM os Conselheiros do Tribunal de Contas ...,"{'multas': [], 'ressarcimentos': [], 'obrigaco...",[],deepseek-v3,cot
3,3,"Vistos, relatados e discutidos estes autos, ac...",NaN,[],deepseek-v3,cot
4,4,DECIDEM os Conselheiros do Tribunal de Contas ...,NaN,[],deepseek-v3,cot
...,...,...,...,...,...,...
861,861,"Vistos, relatados e discutidos estes autos do ...","{'multas': [], 'ressarcimentos': [{'descricao_...","[{'start': 910, 'end': 1352, 'text': 'ressarci...",deepseek-v3,cot
862,862,"Vistos, relatados e discutidos estes autos de ...","{'multas': [], 'ressarcimentos': [{'descricao_...","[{'start': 493, 'end': 683, 'text': 'ressarcim...",deepseek-v3,cot
863,863,"Vistos, relatados e discutidos estes autos da ...",{'multas': [{'descricao_multa': 'Multa no valo...,"[{'start': 887, 'end': 1410, 'text': 'Ressarci...",deepseek-v3,cot
864,864,"Vistos, relatados e discutidos estes autos do ...",{'multas': [{'descricao_multa': 'multa total d...,"[{'start': 588, 'end': 1249, 'text': 'ressarci...",deepseek-v3,cot


In [50]:
df_exp['pred_as_golden'] = df_exp.apply(lambda row: convert_pred_to_golden_format(row, window_size=500, step_size=100, min_score=80), axis=1)
    

AttributeError: 'float' object has no attribute 'items'

In [42]:
all_metrics = {}

for model_name, technique in experiments:
    result_path = get_result_path(model_name, technique)
    if not os.path.exists(result_path):
        print(f"  ⚠ Missing: {model_name}/{technique}")
        continue
    
    df_exp = pd.read_json(result_path)
    df_exp['pred_as_golden'] = df_exp.apply(
        lambda row: convert_pred_to_golden_format(row, window_size=500, step_size=100, min_score=80), axis=1)
    
    print(f"Metrics: {model_name} / {technique}")
    key = f"{model_name}__{technique}"
    all_metrics[key] = calculate_metrics(df_exp, iou_threshold=0.5)
    all_metrics[key]["model"] = model_name
    all_metrics[key]["technique"] = technique

Metrics: gpt-5.4-mini / cot
Metrics: gpt-5.4-nano / cot
Metrics: gemini-2.5-flash / cot


AttributeError: 'NoneType' object has no attribute 'items'

# 8. Results Summary


In [47]:
summary_rows = []
for key, m in all_metrics.items():
    summary_rows.append({
        "Model": m["model"],
        "Technique": m["technique"],
        "Token P": round(m["token_flat"]["precision"], 4),
        "Token R": round(m["token_flat"]["recall"], 4),
        "Token F1": round(m["token_flat"]["f1"], 4),
        "Span P": round(m["iou_agg"]["precision"], 4),
        "Span R": round(m["iou_agg"]["recall"], 4),
        "Span F1": round(m["iou_agg"]["f1"], 4),
    })

df_summary = pd.DataFrame(summary_rows).sort_values("Span F1", ascending=False).reset_index(drop=True)
print(df_summary.to_string(index=False))

           Model Technique  Token P  Token R  Token F1  Span P  Span R  Span F1
    gpt-5.4-nano       cot   0.8146   0.7747    0.7942  0.7154  0.8360   0.7710
    gpt-5.4-mini       cot   0.8010   0.7721    0.7863  0.6824  0.8223   0.7459
gemini-2.5-flash       cot   0.7603   0.7750    0.7676  0.6619  0.8474   0.7433


In [ ]:
# Best technique per model
print("\n=== Best technique per model ===\n")
best_per_model = df_summary.loc[df_summary.groupby("Model")["Span F1"].idxmax()]
print(best_per_model[["Model", "Technique", "Token F1", "Span F1"]].to_string(index=False))


=== Best technique per model ===

       Model Technique  Token F1  Span F1
gpt-5.4-mini  few_shot    0.7879   0.7640
gpt-5.4-nano  few_shot    0.7820   0.7574


In [ ]:
# Best model per technique
print("\n=== Best model per technique ===\n")
best_per_technique = df_summary.loc[df_summary.groupby("Technique")["Span F1"].idxmax()]
print(best_per_technique[["Technique", "Model", "Token F1", "Span F1"]].to_string(index=False))


=== Best model per technique ===

Technique        Model  Token F1  Span F1
 few_shot gpt-5.4-mini    0.7879    0.764


In [ ]:
# Family-level analysis
family_map = {}
for name in models_names:
    if name.startswith("gpt-3") or name.startswith("gpt-4"):
        family_map[name] = "OpenAI 4.x"
    elif name.startswith("gpt-5"):
        family_map[name] = "OpenAI 5.x"
    elif name.startswith("claude"):
        family_map[name] = "Anthropic"
    elif name.startswith("gemini"):
        family_map[name] = "Google"
    elif name.startswith("deepseek"):
        family_map[name] = "DeepSeek"
    elif name.startswith("qwen"):
        family_map[name] = "Qwen"
    elif name.startswith("llama"):
        family_map[name] = "Meta"
    elif name.startswith("mistral"):
        family_map[name] = "Mistral"

df_summary["Family"] = df_summary["Model"].map(family_map)

print("\n=== Average Span F1 by Family (across all techniques) ===\n")
family_avg = df_summary.groupby("Family")["Span F1"].mean().sort_values(ascending=False)
print(family_avg.to_string())


=== Average Span F1 by Family (across all techniques) ===

Family
OpenAI 5.x    0.7607


In [ ]:
# Per-label breakdown for top-10
print("\n=== Per-label IoU F1 for top-10 experiments ===\n")
top_keys = [f"{row['Model']}__{row['Technique']}" for _, row in df_summary.head(10).iterrows()]
for key in top_keys:
    if key not in all_metrics: continue
    m = all_metrics[key]
    print(f"\n{m['model']} / {m['technique']}:")
    for label, vals in m["iou_per_label"].items():
        print(f"  {label}: P={vals['precision']:.4f} R={vals['recall']:.4f} F1={vals['f1']:.4f}")


=== Per-label IoU F1 for top-10 experiments ===


gpt-5.4-mini / few_shot:
  MULTA: P=0.8356 R=0.9059 F1=0.8694
  OBRIGACAO: P=0.6042 R=0.7311 F1=0.6616
  RECOMENDACAO: P=0.6092 R=0.9464 F1=0.7413
  RESSARCIMENTO: P=0.5974 R=0.7419 F1=0.6619

gpt-5.4-nano / few_shot:
  MULTA: P=0.7845 R=0.9010 F1=0.8387
  OBRIGACAO: P=0.6197 R=0.7395 F1=0.6743
  RECOMENDACAO: P=0.5862 R=0.9107 F1=0.7133
  RESSARCIMENTO: P=0.6364 R=0.7903 F1=0.7050


In [ ]:
# Save all metrics
with open("dataset/labeled_data/all_metrics_summary.json", "w", encoding="utf-8") as f:
    json.dump(all_metrics, f, ensure_ascii=False, indent=2, default=str)

df_summary.to_csv("dataset/labeled_data/summary_table.csv", index=False)
print("Results saved to dataset/labeled_data/")

Results saved to dataset/labeled_data/


# 9. Conclusions

This expanded evaluation covers:

- **18 LLMs** across **6 families**: OpenAI (4.x + 5.x), Anthropic, Google, DeepSeek, Qwen, Meta/LLaMA, Mistral
- **8 prompt engineering techniques** from simple prompt swaps to multi-call pipelines
- All models accessed via **OpenRouter** with a single API key

This directly addresses the limitation noted in the original study: *"the evaluation is restricted to models available through Azure OpenAI, excluding strong open-source contenders such as LLaMA 3 or Mistral"*.

The inclusion of DeepSeek (V3 + R1), Qwen 3, LLaMA 4 Maverick, and Mistral models provides a comprehensive open-source vs proprietary comparison for legal NER in Portuguese.
